# 02 — Pipeline de Transformação: Adstock + Saturação de Hill

**Objetivo:** aplicar as transformações não-lineares de mídia antes da regressão.

Roteiro:
1. Por que transformar o spend bruto?
2. Adstock geométrico — conceito e visualização
3. Saturação de Hill — conceito e visualização
4. Sensibilidade dos parâmetros (γ, κ, λ)
5. Busca de parâmetros via random search por canal
6. Aplicar o pipeline a todos os canais e salvar base transformada

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.transformations import (
    geometric_adstock,
    hill_saturation,
    transform_channel,
    transform_all_channels,
    saturation_curve_points,
    adstock_halflife,
    marginal_return,
    random_search_params,
)
from src.model import DEFAULT_CHANNEL_PARAMS

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
print("Módulos carregados.")

In [ ]:
df = pd.read_csv("../data/MMM_Synth_Weekly_Data.csv", parse_dates=["week"])
df = df.sort_values("week").reset_index(drop=True)
spend_cols = [c for c in df.columns if c.startswith("spend_")]
print(f"Dados: {df.shape} | Período: {df['week'].min().date()} → {df['week'].max().date()}")

## 1. Por que transformar o spend bruto?

Usar o spend bruto em uma regressão linear ignora dois fenômenos críticos:

| Fenômeno | Problema ignorado | Solução |
|---|---|---|
| **Carryover** | O efeito de um anúncio persiste além da semana de veiculação | Adstock geométrico |
| **Retornos decrescentes** | Dobrar o budget não dobra os assinantes | Saturação de Hill |

Sem essas transformações, o modelo subestima canais com alto carryover (YouTube) e superestima canais saturados.

## 2. Adstock Geométrico

In [ ]:
# Exemplo: impacto de uma única semana de spend em diferentes λ
T       = 20
spend_pulse = np.zeros(T)
spend_pulse[0] = 100_000  # R$ 100k em uma única semana

lambdas = [0.05, 0.20, 0.40, 0.60, 0.80]
fig, ax = plt.subplots(figsize=(12, 4))

for lam in lambdas:
    ads = geometric_adstock(spend_pulse, lam)
    hl  = adstock_halflife(lam)
    ax.plot(ads / 1000, marker="o", markersize=4,
            label=f"λ = {lam:.2f}  (meia-vida: {hl:.1f} sem.)")

ax.set_title("Adstock Geométrico — Decay de um Pulso de R$ 100k", fontweight="bold")
ax.set_xlabel("Semanas após o investimento")
ax.set_ylabel("Adstock (R$ mil)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/adstock_decay_example.png", dpi=130, bbox_inches="tight")
plt.show()

print("\nMeia-vida por λ:")
for lam in lambdas:
    print(f"  λ = {lam:.2f}  → meia-vida: {adstock_halflife(lam):.2f} semanas")

## 3. Saturação de Hill

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Efeito de γ (mantendo κ fixo)
kappa_fixo = 200_000
gammas = [0.3, 0.6, 0.8, 1.0, 1.5]
for g in gammas:
    x, s = saturation_curve_points(g, kappa_fixo)
    axes[0].plot(x / 1000, s, label=f"γ = {g}")
axes[0].axvline(kappa_fixo / 1000, color="gray", lw=1, linestyle="--")
axes[0].text(kappa_fixo / 1000 * 1.02, 0.1, "κ (meia-saturação)", color="gray", fontsize=8)
axes[0].set_xlabel("Adstock (R$ mil)")
axes[0].set_ylabel("Saturação [0, 1)")
axes[0].set_title("Efeito de γ (κ fixo)", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Efeito de κ (mantendo γ fixo)
gamma_fixo = 0.8
kappas = [50_000, 100_000, 200_000, 400_000]
for k in kappas:
    x, s = saturation_curve_points(gamma_fixo, k, x_max_multiplier=5)
    axes[1].plot(x / 1000, s, label=f"κ = {k/1000:.0f}k")
axes[1].set_xlabel("Adstock (R$ mil)")
axes[1].set_ylabel("Saturação [0, 1)")
axes[1].set_title("Efeito de κ (γ fixo = 0.8)", fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

fig.suptitle("Função de Saturação de Hill — Sensibilidade dos Parâmetros",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/hill_saturation_sensitivity.png", dpi=130, bbox_inches="tight")
plt.show()

## 4. Retorno Marginal — onde está a fronteira de eficiência?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

for ch_name, params in DEFAULT_CHANNEL_PARAMS.items():
    kap = params["kappa"]
    gam = params["gamma"]
    x   = np.linspace(1, kap * 3, 300)
    mr  = np.array([marginal_return(xi, gam, kap) for xi in x])
    ax.plot(x / 1000, mr, label=ch_name.split(" ")[0] + " " + ch_name.split(" ")[-1])

ax.set_xlabel("Adstock (R$ mil)")
ax.set_ylabel("dS/dx — Retorno Marginal")
ax.set_title("Retorno Marginal da Saturação de Hill por Canal", fontweight="bold")
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Curvas de Saturação por Canal (Parâmetros Calibrados)

In [ ]:
n_ch  = len(DEFAULT_CHANNEL_PARAMS)
ncols = 3
nrows = (n_ch + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
axes_flat = axes.flatten()
colors    = sns.color_palette("tab10", n_ch)

for i, (ch_name, params) in enumerate(DEFAULT_CHANNEL_PARAMS.items()):
    ax  = axes_flat[i]
    gam = params["gamma"]
    kap = params["kappa"]
    lam = params["lambda_decay"]

    x, s = saturation_curve_points(gam, kap, x_max_multiplier=2.5)

    # Spend médio real do canal
    col_key   = ch_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    spend_col = f"spend_{col_key}"
    avg_spend = df[spend_col].mean() if spend_col in df.columns else kap

    ax.plot(x / 1000, s, color=colors[i], lw=2)
    ax.axvline(avg_spend / 1000, color="gray", lw=1, linestyle="--", label=f"Spend médio")
    ax.axhline(0.5, color="lightgray", lw=0.8, linestyle=":")
    ax.axvline(kap / 1000, color="salmon", lw=1, linestyle=":", label=f"κ = {kap/1000:.0f}k")

    ax.set_title(ch_name, fontsize=9, fontweight="bold")
    ax.set_xlabel("Adstock (R$ mil)", fontsize=8)
    ax.set_ylabel("Saturação", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)
    ax.text(0.05, 0.92,
            f"λ={lam} | γ={gam} | κ={kap/1000:.0f}k",
            transform=ax.transAxes, fontsize=7, color="#333")

# Esconder eixos extras
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Curvas de Saturação — Parâmetros Calibrados por Canal",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../outputs/saturation_curves_by_channel.png", dpi=130, bbox_inches="tight")
plt.show()

## 6. Aplicar Pipeline e Salvar Base Transformada

In [ ]:
# Aplicar transformações a todos os canais
df_media = transform_all_channels(df, DEFAULT_CHANNEL_PARAMS)

# Juntar com controles e variável dependente
ctrl_cols = ["week", "seasonality_index", "trend", "price_hike_flag", "new_subscribers"]
df_transformed = pd.concat([df[ctrl_cols], df_media], axis=1)

# Salvar
out_path = "../data/MMM_Transformed_Weekly_Data.csv"
df_transformed.to_csv(out_path, index=False)

print(f"Base transformada salva: {out_path}")
print(f"Shape: {df_transformed.shape}")
df_transformed.head(3)

In [ ]:
# Visualizar spend bruto vs. adstock transformado para um canal
ch_ex    = "paid_search_google_"
spend_ex = df[f"spend_{ch_ex}"].values
trans_ex = df_transformed[f"media_{ch_ex}"].values

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df["week"], spend_ex / 1000, color="#0f3460", lw=1.3)
axes[0].set_ylabel("Spend Bruto (R$ mil)")
axes[0].set_title("Paid Search (Google) — Spend Bruto vs. Após Adstock + Hill", fontweight="bold")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["week"], trans_ex, color="#e94560", lw=1.3)
axes[1].set_ylabel("Saturação [0, 1)")
axes[1].set_xlabel("Semana")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n➡ Próximo notebook: 03_mmm_model_hac.ipynb — Ajuste OLS + HAC")